# Análise Estatística de Séries Temporais: Decomposição Clássica, Filtros Lineares e Suavização Exponencial
## Disciplina: Ciência de Dados | Gabarito Resolvido
### Prof. André Souza

---

### Fundamentação Teórica e Metodologia

Uma série temporal é definida matematicamente como uma realização de um processo estocástico indexado no tempo, denotada por ${X_t, t \in T}$, onde $T$ é um conjunto discreto de índices temporais regularmente espaçados.

Neste módulo prático, analisaremos as propriedades dinâmicas de 19 conjuntos de dados empíricos clássicos, contemplando fenômenos geofísicos, atmosféricos, industriais, macroeconômicos e do mercado financeiro.

#### 1. Decomposição Estrutural Clássica
O objetivo da decomposição é isolar as componentes não observáveis da série temporal:
* **Tendência-Ciclo ($T_t$)**: Movimento secular de longo prazo ou ciclos plurianuais da série.
* **Sazonalidade ($S_t$)**: Padrões periódicos determinísticos ou estocásticos com período fixo $s$ tal que $S_{t+s} = S_t$.
* **Resíduo Aleatório / Ruído Estocástico ($R_t$ ou $\epsilon_t$)**: Componente irregular remanescente, idealmente com características de ruído branco ($\mathbb{E}[\epsilon_t]=0$ e covariância nula para defasagens $k \neq 0$).

Os dois modelos canônicos de decomposição implementados são:
$$\text{Modelo Aditivo: } Y_t = T_t + S_t + R_t$$
$$\text{Modelo Multiplicativo: } Y_t = T_t \times S_t \times R_t$$

O modelo aditivo é indicado quando a amplitude da variação sazonal independe do patamar da série. O modelo multiplicativo é indicado quando as variações sazonais e o ruído são proporcionais ao nível médio da tendência.

#### 2. Filtros Lineares: Média Móvel Simples (SMA)
A Média Móvel Simples atua como um filtro passa-baixa (*low-pass filter*), atenuando as componentes de alta frequência (ruído) e preservando a tendência subjacente:
$$SMA_k(t) = \frac{1}{k} \sum_{i=0}^{k-1} Y_{t-i}$$
Quando o tamanho da janela $k$ é fixado exatamente no período da sazonalidade (ex.: $k=12$ para periodicidade mensal ou $k=24$ para dados horários), o filtro elimina integralmente a sazonalidade estrita.

#### 3. Suavização Exponencial: Média Móvel Ponderada Exponencialmente (EWMA)
Ao contrário do filtro retangular da SMA, a formulação EWMA atribui pesos que decrescem exponencialmente em relação à distância temporal:
$$\hat{Y}_t = \alpha Y_t + (1 - \alpha) \hat{Y}_{t-1}, \quad 0 < \alpha \le 1$$
O parâmetro de amortecimento $\alpha$ relaciona-se diretamente com o *span* efetivo $s$ através de:
$$\alpha = \frac{2}{s + 1}$$
A formulação EWMA confere menor defasagem temporal (*phase lag*) diante de quebras estruturais ou inflexões na trajetória da série.


---
## 1. Módulo de Carregamento e Rotinas de Visualização Técnica

Execute a célula abaixo para importar as bibliotecas e disponibilizar as rotinas analíticas padronizadas:
* `BASE_URL`: Endereço base remoto no repositório GitHub para importação direta dos arquivos `.csv`.
* `carregar_serie(nome_arquivo, col_data, col_valor, freq=None)`: Utilitário para leitura, tratamento de índices e sanitização de dados.
* `plotar_sma_ewma(serie, janela_sma, span_ewma, titulo, ylabel, unidade)`: Rotina pronta para visualização gráfica das médias móveis.
* `plotar_decomposicao(serie, periodo, modelo, titulo)`: Rotina pronta para visualização da decomposição estrutural em 4 painéis.


In [ ]:
import os
import urllib.parse
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

# Configurações visuais padronizadas
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

BASE_URL = "https://raw.githubusercontent.com/profAndreSouza/Material/refs/heads/main/Ciencia%20de%20Dados/materiais/time_series_pam/"

def carregar_serie(nome_arquivo, col_data, col_valor, freq=None):
    """
    Carrega o dataset a partir do repositório remoto (ou pasta local de materiais/time_series_pam),
    trata valores nulos/legados ('na') e retorna uma pd.Series indexada temporalmente.
    """
    url = BASE_URL + urllib.parse.quote(nome_arquivo)
    caminho_local_1 = os.path.join("materiais", "time_series_pam", nome_arquivo)
    caminho_local_2 = os.path.join("materiais", nome_arquivo)

    try:
        df = pd.read_csv(url)
    except Exception:
        if os.path.exists(caminho_local_1):
            df = pd.read_csv(caminho_local_1)
        elif os.path.exists(caminho_local_2):
            df = pd.read_csv(caminho_local_2)
        else:
            raise FileNotFoundError(f"Dataset {nome_arquivo} não encontrado via URL ou disco local.")

    try:
        df[col_data] = pd.to_datetime(df[col_data])
    except Exception:
        df[col_data] = pd.to_numeric(df[col_data], errors='coerce')

    df = df.set_index(col_data).sort_index()
    serie = pd.to_numeric(df[col_valor], errors='coerce').dropna()

    if freq:
        try:
            serie = serie.asfreq(freq)
        except Exception:
            pass

    return serie

def plotar_sma_ewma(serie, janela_sma, span_ewma, titulo, ylabel="Valor", unidade="Períodos"):
    """
    Plota a série observada sobreposta pela Média Móvel Simples (SMA) e Média Móvel Exponencial (EWMA).
    """
    sma = serie.rolling(window=janela_sma, min_periods=1).mean()
    ewma = serie.ewm(span=span_ewma, adjust=False).mean()

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(serie.index, serie.values, label='Série Observada (Original)', color='#4A5568', alpha=0.45, lw=1.2)
    ax.plot(serie.index, sma.values, label=f'SMA (Janela = {janela_sma} {unidade})', color='#1A365D', lw=2.2)
    ax.plot(serie.index, ewma.values, label=f'EWMA (Span = {span_ewma} {unidade})', color='#C53030', lw=2.0, linestyle='--')

    ax.set_title(titulo, fontweight='bold', pad=10)
    ax.set_xlabel('Eixo Temporal', fontweight='bold')
    ax.set_ylabel(ylabel, fontweight='bold')
    ax.legend(loc='best', frameon=True)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plotar_decomposicao(serie, periodo, modelo='additive', titulo="Decomposição Estrutural"):
    """
    Executa e plota os 4 painéis da decomposição clássica: Observado, Tendência, Sazonalidade e Resíduo.
    """
    s_interp = serie.interpolate(method='linear').dropna()
    decomp = seasonal_decompose(s_interp, model=modelo, period=periodo)

    fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)

    axes[0].plot(decomp.observed.index, decomp.observed.values, color='#1A202C', lw=1.4)
    axes[0].set_ylabel('Observado', fontweight='bold')
    axes[0].set_title(f"{titulo} | Modelo: {modelo.capitalize()} (Período s = {periodo})", fontweight='bold', pad=8)

    axes[1].plot(decomp.trend.index, decomp.trend.values, color='#2B6CB0', lw=2.0)
    axes[1].set_ylabel('Tendência', fontweight='bold')

    axes[2].plot(decomp.seasonal.index, decomp.seasonal.values, color='#276749', lw=1.4)
    axes[2].set_ylabel('Sazonalidade', fontweight='bold')

    axes[3].scatter(decomp.resid.index, decomp.resid.values, color='#9B2C2C', s=12, alpha=0.7)
    axes[3].axhline(0 if modelo == 'additive' else 1, color='black', linestyle=':', lw=1.2, alpha=0.7)
    axes[3].set_ylabel('Resíduo', fontweight='bold')
    axes[3].set_xlabel('Eixo Temporal', fontweight='bold')

    for ax in axes:
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

print("Módulo de suporte inicializado. Funções 'carregar_serie', 'plotar_sma_ewma' e 'plotar_decomposicao' prontas para uso.")


---
## Exercício 01: Temperatura Média Mensal — Estações Cananéia e Ubatuba (SP)
### Base de Dados: `a1_temperatura.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: Morettin & Toloi (Análise de Séries Temporais). Mensal (1976 a 1985 — 120 observações). Variáveis: Ano, Cananeia, Ubatuba.
* **Coluna Temporal**: `Ano`
* **Variável de Análise Principal**: `Cananeia`
* **Parâmetros Sugeridos**: Janela SMA = `12` meses, Span EWMA = `6` meses, Período Sazonal $s$ = `12`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `a1_temperatura.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 01: a1_temperatura.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_1 = carregar_serie(
    nome_arquivo="a1_temperatura.csv",
    col_data="Ano",
    col_valor="Cananeia"
)

print(f"Dataset: a1_temperatura.csv")
print(f"Total de registros válidos: {len(serie_1)}")
print(f"Intervalo temporal: [{serie_1.index[0]}] até [{serie_1.index[-1]}]")
print("Primeiros registros observados:\n", serie_1.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_1,
    janela_sma=12,
    span_ewma=6,
    titulo="Exercício 01 — Suavização Temporal (SMA vs EWMA): Temperatura Média Mensal — Estações Cananéia e Ubatuba (SP)",
    ylabel="Cananeia",
    unidade="meses"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_1,
    periodo=12,
    modelo='additive',
    titulo="Exercício 01 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 01):
1. Analise o comportamento do filtro SMA com janela k=12: a componente sazonal intra-anual foi integralmente atenuada? Explique matematicamente o motivo.
2. A série decomposta indica tendência monotônica de aquecimento ou estabilidade estacionária na média?
3. Avalie se a hipótese aditiva é compatível com a estabilidade de variância observada na componente residual.


---
## Exercício 02: Atividade Magnética Solar — Série Anual de Manchas Solares (1749 a 1924)
### Base de Dados: `a2_MANCHAS.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: Observatório de Zurique / Wolf Sunspot Numbers. Anual (1749 a 1924 — 176 observações). Variáveis: Ano, manchas.
* **Coluna Temporal**: `Ano`
* **Variável de Análise Principal**: `manchas`
* **Parâmetros Sugeridos**: Janela SMA = `11` anos, Span EWMA = `6` anos, Período Sazonal $s$ = `11`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `a2_MANCHAS.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 02: a2_MANCHAS.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_2 = carregar_serie(
    nome_arquivo="a2_MANCHAS.csv",
    col_data="Ano",
    col_valor="manchas"
)

print(f"Dataset: a2_MANCHAS.csv")
print(f"Total de registros válidos: {len(serie_2)}")
print(f"Intervalo temporal: [{serie_2.index[0]}] até [{serie_2.index[-1]}]")
print("Primeiros registros observados:\n", serie_2.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_2,
    janela_sma=11,
    span_ewma=6,
    titulo="Exercício 02 — Suavização Temporal (SMA vs EWMA): Atividade Magnética Solar — Série Anual de Manchas Solares (1749 a 1924)",
    ylabel="manchas",
    unidade="anos"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_2,
    periodo=11,
    modelo='additive',
    titulo="Exercício 02 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 02):
1. Justifique a parametrização de k=11 para a janela da SMA com base na dinâmica do ciclo de Schwabe.
2. Compare a defasagem dinâmica da EWMA com relação à SMA nos momentos de rápida ascensão dos picos de atividade solar.
3. Como a variabilidade na amplitude dos ciclos afeta a magnitude dos resíduos estimados na decomposição?


---
## Exercício 03: Índice Pluviométrico Anual — Fortaleza, Ceará (1849 a 1997)
### Base de Dados: `a3_fortaleza.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: FUNCEME / Morettin & Toloi. Anual (1849 a 1997 — 149 observações). Variáveis: Ano, Fort.
* **Coluna Temporal**: `Ano`
* **Variável de Análise Principal**: `Fort`
* **Parâmetros Sugeridos**: Janela SMA = `10` anos, Span EWMA = `5` anos, Período Sazonal $s$ = `10`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `a3_fortaleza.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 03: a3_fortaleza.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_3 = carregar_serie(
    nome_arquivo="a3_fortaleza.csv",
    col_data="Ano",
    col_valor="Fort"
)

print(f"Dataset: a3_fortaleza.csv")
print(f"Total de registros válidos: {len(serie_3)}")
print(f"Intervalo temporal: [{serie_3.index[0]}] até [{serie_3.index[-1]}]")
print("Primeiros registros observados:\n", serie_3.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_3,
    janela_sma=10,
    span_ewma=5,
    titulo="Exercício 03 — Suavização Temporal (SMA vs EWMA): Índice Pluviométrico Anual — Fortaleza, Ceará (1849 a 1997)",
    ylabel="Fort",
    unidade="anos"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_3,
    periodo=10,
    modelo='additive',
    titulo="Exercício 03 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 03):
1. A série evidencia tendência secular persistente de estiagem ou flutuações estacionárias em torno da média histórica?
2. Identifique nos gráficos as anomalias negativas extremas correspondentes a secas históricas documentadas (ex.: período 1877–1879).
3. A componente residual apresenta propriedades compatíveis com ruído não correlacionado?


---
## Exercício 04: Precipitação Pluviométrica Mensal — Lavras, Minas Gerais (1966 a 1997)
### Base de Dados: `a3_LAVRAS.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: Estação Climatológica de Lavras / UFLA. Mensal (1966 a 1997 — 384 observações). Variáveis: Ano, Precipitacao.
* **Coluna Temporal**: `Ano`
* **Variável de Análise Principal**: `Precipitacao`
* **Parâmetros Sugeridos**: Janela SMA = `12` meses, Span EWMA = `6` meses, Período Sazonal $s$ = `12`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `a3_LAVRAS.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 04: a3_LAVRAS.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_4 = carregar_serie(
    nome_arquivo="a3_LAVRAS.csv",
    col_data="Ano",
    col_valor="Precipitacao"
)

print(f"Dataset: a3_LAVRAS.csv")
print(f"Total de registros válidos: {len(serie_4)}")
print(f"Intervalo temporal: [{serie_4.index[0]}] até [{serie_4.index[-1]}]")
print("Primeiros registros observados:\n", serie_4.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_4,
    janela_sma=12,
    span_ewma=6,
    titulo="Exercício 04 — Suavização Temporal (SMA vs EWMA): Precipitação Pluviométrica Mensal — Lavras, Minas Gerais (1966 a 1997)",
    ylabel="Precipitacao",
    unidade="meses"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_4,
    periodo=12,
    modelo='additive',
    titulo="Exercício 04 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 04):
1. Observe a amplitude estimada da componente sazonal e compare a magnitude média dos meses de inverno com os meses de verão.
2. A aplicação de SMA_12 removeu a oscilação sazonal sem distorcer o nível pluviométrico de longo prazo?
3. O volume pluviométrico total acumulado apresenta tendência sistemática de aumento ou decréscimo interanual?


---
## Exercício 05: Série Temporal de Concentração Mensal de Ozônio Atmosférico
### Base de Dados: `a4_OZONIO.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: Física atmosférica / Morettin & Toloi. Mensal (1956 a 1970 — 180 observações). Variáveis: Ano, Ozonio.
* **Coluna Temporal**: `Ano`
* **Variável de Análise Principal**: `Ozonio`
* **Parâmetros Sugeridos**: Janela SMA = `12` meses, Span EWMA = `6` meses, Período Sazonal $s$ = `12`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `a4_OZONIO.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 05: a4_OZONIO.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_5 = carregar_serie(
    nome_arquivo="a4_OZONIO.csv",
    col_data="Ano",
    col_valor="Ozonio"
)

print(f"Dataset: a4_OZONIO.csv")
print(f"Total de registros válidos: {len(serie_5)}")
print(f"Intervalo temporal: [{serie_5.index[0]}] até [{serie_5.index[-1]}]")
print("Primeiros registros observados:\n", serie_5.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_5,
    janela_sma=12,
    span_ewma=6,
    titulo="Exercício 05 — Suavização Temporal (SMA vs EWMA): Série Temporal de Concentração Mensal de Ozônio Atmosférico",
    ylabel="Ozonio",
    unidade="meses"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_5,
    periodo=12,
    modelo='additive',
    titulo="Exercício 05 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 05):
1. Há evidência estatística de quebra estrutural ou tendência monotônica no teor médio de ozônio no período?
2. A componente sazonal apresenta comportamento simétrico ao longo do ano geofísico?
3. Como os resíduos se distribuem temporalmente: há presença de agrupamento de volatilidade?


---
## Exercício 06: Demanda e Consumo Mensal de Energia Elétrica (1968 a 1979)
### Base de Dados: `a5_ENERGIA.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: Sistema Elétrico Brasileiro / Morettin & Toloi. Mensal (1968 a 1979 — 141 observações). Variáveis: Ano, Energia.
* **Coluna Temporal**: `Ano`
* **Variável de Análise Principal**: `Energia`
* **Parâmetros Sugeridos**: Janela SMA = `12` meses, Span EWMA = `6` meses, Período Sazonal $s$ = `12`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `a5_ENERGIA.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 06: a5_ENERGIA.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_6 = carregar_serie(
    nome_arquivo="a5_ENERGIA.csv",
    col_data="Ano",
    col_valor="Energia"
)

print(f"Dataset: a5_ENERGIA.csv")
print(f"Total de registros válidos: {len(serie_6)}")
print(f"Intervalo temporal: [{serie_6.index[0]}] até [{serie_6.index[-1]}]")
print("Primeiros registros observados:\n", serie_6.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_6,
    janela_sma=12,
    span_ewma=6,
    titulo="Exercício 06 — Suavização Temporal (SMA vs EWMA): Demanda e Consumo Mensal de Energia Elétrica (1968 a 1979)",
    ylabel="Energia",
    unidade="meses"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_6,
    periodo=12,
    modelo='additive',
    titulo="Exercício 06 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 06):
1. Por que a formulação multiplicativa (Y_t = T_t * S_t * R_t) é teoricamente superior à aditiva quando a amplitude sazonal cresce proporcionalmente ao nível da série?
2. Compare a heterocedasticidade dos resíduos sob ambas as hipóteses (aditiva vs multiplicativa).
3. Como a EWMA responde ao crescimento exponencial da demanda comparada ao atraso linear (lag) da SMA?


---
## Exercício 07: Concentração Diária de Poluentes Atmosféricos — São Paulo (1997)
### Base de Dados: `a6_poluicao.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: CETESB. Diária (ano de 1997 — 365 observações). Variáveis: DATA, RES65, PM10, so2, co, o3, no2.
* **Coluna Temporal**: `DATA`
* **Variável de Análise Principal**: `PM10`
* **Parâmetros Sugeridos**: Janela SMA = `7` dias, Span EWMA = `7` dias, Período Sazonal $s$ = `7`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `a6_poluicao.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 07: a6_poluicao.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_7 = carregar_serie(
    nome_arquivo="a6_poluicao.csv",
    col_data="DATA",
    col_valor="PM10"
)

print(f"Dataset: a6_poluicao.csv")
print(f"Total de registros válidos: {len(serie_7)}")
print(f"Intervalo temporal: [{serie_7.index[0]}] até [{serie_7.index[-1]}]")
print("Primeiros registros observados:\n", serie_7.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_7,
    janela_sma=7,
    span_ewma=7,
    titulo="Exercício 07 — Suavização Temporal (SMA vs EWMA): Concentração Diária de Poluentes Atmosféricos — São Paulo (1997)",
    ylabel="PM10",
    unidade="dias"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_7,
    periodo=7,
    modelo='additive',
    titulo="Exercício 07 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 07):
1. Por que a periodicidade s=7 é a escolha analítica adequada para modelar emissões antrópicas urbanas?
2. Em qual trimestre do ano a tendência do material particulado (PM10) atinge valores críticos? Qual a relação com as condições meteorológicas de inverno?
3. Substitua a variável por 'co' ou 'no2' e verifique se a componente sazonal semanal apresenta queda acentuada aos fins de semana.


---
## Exercício 08: Monitoramento Micrometeorológico Diário — Temperatura e Umidade (1997)
### Base de Dados: `a7_atmosfera.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: IAG/USP. Diária (1997 — 365 observações). Variáveis: Time, temperatura, umidade.
* **Coluna Temporal**: `Time`
* **Variável de Análise Principal**: `temperatura`
* **Parâmetros Sugeridos**: Janela SMA = `7` dias, Span EWMA = `7` dias, Período Sazonal $s$ = `30`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `a7_atmosfera.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 08: a7_atmosfera.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_8 = carregar_serie(
    nome_arquivo="a7_atmosfera.csv",
    col_data="Time",
    col_valor="temperatura"
)

print(f"Dataset: a7_atmosfera.csv")
print(f"Total de registros válidos: {len(serie_8)}")
print(f"Intervalo temporal: [{serie_8.index[0]}] até [{serie_8.index[-1]}]")
print("Primeiros registros observados:\n", serie_8.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_8,
    janela_sma=7,
    span_ewma=7,
    titulo="Exercício 08 — Suavização Temporal (SMA vs EWMA): Monitoramento Micrometeorológico Diário — Temperatura e Umidade (1997)",
    ylabel="temperatura",
    unidade="dias"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_8,
    periodo=30,
    modelo='additive',
    titulo="Exercício 08 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 08):
1. Como a SMA semanal filtra oscilações de alta frequência associadas à passagem de sistemas frontais frios?
2. A tendência estimada descreve o ciclo termodinâmico anual das estações climáticas?
3. Ao inverter a análise para a umidade, a trajetória da componente de tendência confirma a correlação física inversa com a temperatura no inverno?


---
## Exercício 09: Trajetória Secular do Produto Interno Bruto Brasileiro (1861 a 1986)
### Base de Dados: `a8_a_pibanual.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: IPEA / IBGE / Morettin & Toloi. Anual (1861 a 1986 — 126 observações). Variáveis: ano, pib1949, Lpib1949, DLpib1949.
* **Coluna Temporal**: `ano`
* **Variável de Análise Principal**: `pib1949`
* **Parâmetros Sugeridos**: Janela SMA = `10` anos, Span EWMA = `5` anos, Período Sazonal $s$ = `10`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `a8_a_pibanual.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 09: a8_a_pibanual.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_9 = carregar_serie(
    nome_arquivo="a8_a_pibanual.csv",
    col_data="ano",
    col_valor="pib1949"
)

print(f"Dataset: a8_a_pibanual.csv")
print(f"Total de registros válidos: {len(serie_9)}")
print(f"Intervalo temporal: [{serie_9.index[0]}] até [{serie_9.index[-1]}]")
print("Primeiros registros observados:\n", serie_9.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_9,
    janela_sma=10,
    span_ewma=5,
    titulo="Exercício 09 — Suavização Temporal (SMA vs EWMA): Trajetória Secular do Produto Interno Bruto Brasileiro (1861 a 1986)",
    ylabel="pib1949",
    unidade="anos"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_9,
    periodo=10,
    modelo='additive',
    titulo="Exercício 09 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 09):
1. A série em nível do PIB real é não-estacionária em média. Qual a ordem de integração evidente visualmente?
2. Aplique as rotinas analíticas na coluna DLpib1949 (taxa de crescimento): o processo resultante aproxima-se de uma estrutura estacionária?
3. Identifique o impacto de choques exógenos nos resíduos da taxa de crescimento (ex.: Crise de 1929 e Segunda Guerra Mundial).


---
## Exercício 10: Índice de Produção Industrial (IPI) — Setor de Alimentos (1985 a 2000)
### Base de Dados: `a8_b_IPI.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: PIM-IBGE / Banco Central do Brasil. Mensal (1985 a 2000 — 186 observações). Variáveis: Time, ipialiment.
* **Coluna Temporal**: `Time`
* **Variável de Análise Principal**: `ipialiment`
* **Parâmetros Sugeridos**: Janela SMA = `12` meses, Span EWMA = `6` meses, Período Sazonal $s$ = `12`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `a8_b_IPI.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 10: a8_b_IPI.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_10 = carregar_serie(
    nome_arquivo="a8_b_IPI.csv",
    col_data="Time",
    col_valor="ipialiment"
)

print(f"Dataset: a8_b_IPI.csv")
print(f"Total de registros válidos: {len(serie_10)}")
print(f"Intervalo temporal: [{serie_10.index[0]}] até [{serie_10.index[-1]}]")
print("Primeiros registros observados:\n", serie_10.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_10,
    janela_sma=12,
    span_ewma=6,
    titulo="Exercício 10 — Suavização Temporal (SMA vs EWMA): Índice de Produção Industrial (IPI) — Setor de Alimentos (1985 a 2000)",
    ylabel="ipialiment",
    unidade="meses"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_10,
    periodo=12,
    modelo='additive',
    titulo="Exercício 10 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 10):
1. A produção de bens não-duráveis de consumo (alimentos) apresenta menor amplitude de oscilação cíclica do que bens de capital? Por quê?
2. Como a estabilização macroeconômica pós-1994 (Plano Real) alterou o nível e a volatilidade da tendência do setor?
3. A EWMA permitiu identificar a mudança de patamar com menor tempo de resposta do que o filtro SMA?


---
## Exercício 11: Produção Física Industrial Geral — PFI Brasil (1991 a 2000)
### Base de Dados: `a8_c_PFI.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: IBGE. Mensal (1991 a 2000 — 114 observações). Variáveis: Time, PFI.
* **Coluna Temporal**: `Time`
* **Variável de Análise Principal**: `PFI`
* **Parâmetros Sugeridos**: Janela SMA = `12` meses, Span EWMA = `6` meses, Período Sazonal $s$ = `12`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `a8_c_PFI.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 11: a8_c_PFI.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_11 = carregar_serie(
    nome_arquivo="a8_c_PFI.csv",
    col_data="Time",
    col_valor="PFI"
)

print(f"Dataset: a8_c_PFI.csv")
print(f"Total de registros válidos: {len(serie_11)}")
print(f"Intervalo temporal: [{serie_11.index[0]}] até [{serie_11.index[-1]}]")
print("Primeiros registros observados:\n", serie_11.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_11,
    janela_sma=12,
    span_ewma=6,
    titulo="Exercício 11 — Suavização Temporal (SMA vs EWMA): Produção Física Industrial Geral — PFI Brasil (1991 a 2000)",
    ylabel="PFI",
    unidade="meses"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_11,
    periodo=12,
    modelo='additive',
    titulo="Exercício 11 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 11):
1. A componente sazonal estimada confirma a concentração de atividade manufatureira no segundo semestre para abastecimento do varejo?
2. Identifique os impactos recessivos das crises financeiras internacionais (Crise Asiática de 1997 e Crise Russa de 1998) na trajetória da tendência.
3. Como se comportam os resíduos em momentos de descontinuidade de política cambial (janeiro de 1999)?


---
## Exercício 12: Índice de Produção Industrial do Setor de Bebidas (1985 a 2000)
### Base de Dados: `a8_d_Bebida.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: IBGE / Morettin & Toloi. Mensal (1985 a 2000 — 187 observações). Variáveis: Time, Bebida.
* **Coluna Temporal**: `Time`
* **Variável de Análise Principal**: `Bebida`
* **Parâmetros Sugeridos**: Janela SMA = `12` meses, Span EWMA = `6` meses, Período Sazonal $s$ = `12`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `a8_d_Bebida.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 12: a8_d_Bebida.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_12 = carregar_serie(
    nome_arquivo="a8_d_Bebida.csv",
    col_data="Time",
    col_valor="Bebida"
)

print(f"Dataset: a8_d_Bebida.csv")
print(f"Total de registros válidos: {len(serie_12)}")
print(f"Intervalo temporal: [{serie_12.index[0]}] até [{serie_12.index[-1]}]")
print("Primeiros registros observados:\n", serie_12.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_12,
    janela_sma=12,
    span_ewma=6,
    titulo="Exercício 12 — Suavização Temporal (SMA vs EWMA): Índice de Produção Industrial do Setor de Bebidas (1985 a 2000)",
    ylabel="Bebida",
    unidade="meses"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_12,
    periodo=12,
    modelo='additive',
    titulo="Exercício 12 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 12):
1. Qual a razão percentual entre o pico sazonal de dezembro e o ponto de mínimo sazonal no meio do ano?
2. Demonstre por que a aplicação de SMA_12 é indispensável para que o gestor de supply chain enxergue a demanda real desprovida da ilusão sazonal do verão.
3. Há evidência de alteração estrutural no comportamento sazonal ao longo dos 15 anos observados?


---
## Exercício 13: Índice de Custo de Vida (ICV) em São Paulo — Período 1970 a 1980
### Base de Dados: `a10_ICV.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: DIEESE. Mensal (1970 a 1980 — 125 observações). Variáveis: Time, ICV.
* **Coluna Temporal**: `Time`
* **Variável de Análise Principal**: `ICV`
* **Parâmetros Sugeridos**: Janela SMA = `12` meses, Span EWMA = `6` meses, Período Sazonal $s$ = `12`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `a10_ICV.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 13: a10_ICV.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_13 = carregar_serie(
    nome_arquivo="a10_ICV.csv",
    col_data="Time",
    col_valor="ICV"
)

print(f"Dataset: a10_ICV.csv")
print(f"Total de registros válidos: {len(serie_13)}")
print(f"Intervalo temporal: [{serie_13.index[0]}] até [{serie_13.index[-1]}]")
print("Primeiros registros observados:\n", serie_13.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_13,
    janela_sma=12,
    span_ewma=6,
    titulo="Exercício 13 — Suavização Temporal (SMA vs EWMA): Índice de Custo de Vida (ICV) em São Paulo — Período 1970 a 1980",
    ylabel="ICV",
    unidade="meses"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_13,
    periodo=12,
    modelo='additive',
    titulo="Exercício 13 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 13):
1. Por que uma série de índice cumulativo de preços não deve ser considerada estacionária sob nenhuma hipótese linear?
2. Em um processo com aceleração inflacionária convexa, qual o impacto da defasagem temporal de filtros lineares simétricos?
3. O que ocorreria se a série fosse diferenciada na forma de taxa de variação percentual mensal antes da decomposição?


---
## Exercício 14: Indicador Agregado de Consumo das Famílias (1984 a 1996)
### Base de Dados: `a11_CONSUMO.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: Morettin & Toloi. Mensal (1984 a 1996 — 153 observações). Variáveis: data, consumo.
* **Coluna Temporal**: `data`
* **Variável de Análise Principal**: `consumo`
* **Parâmetros Sugeridos**: Janela SMA = `12` meses, Span EWMA = `6` meses, Período Sazonal $s$ = `12`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `a11_CONSUMO.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 14: a11_CONSUMO.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_14 = carregar_serie(
    nome_arquivo="a11_CONSUMO.csv",
    col_data="data",
    col_valor="consumo"
)

print(f"Dataset: a11_CONSUMO.csv")
print(f"Total de registros válidos: {len(serie_14)}")
print(f"Intervalo temporal: [{serie_14.index[0]}] até [{serie_14.index[-1]}]")
print("Primeiros registros observados:\n", serie_14.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_14,
    janela_sma=12,
    span_ewma=6,
    titulo="Exercício 14 — Suavização Temporal (SMA vs EWMA): Indicador Agregado de Consumo das Famílias (1984 a 1996)",
    ylabel="consumo",
    unidade="meses"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_14,
    periodo=12,
    modelo='additive',
    titulo="Exercício 14 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 14):
1. Localize visualmente as quebras de patamar resultantes do Plano Cruzado (1986) e do Plano Real (1994).
2. A EWMA foi capaz de se ajustar mais prontamente à expansão de consumo pós-estabilização do que a média móvel simples?
3. Como as intervenções de congelamento de preços e controle de liquidez se manifestam na dispersão dos resíduos da decomposição?


---
## Exercício 15: Cotações Diárias das Ações da Globo Cabo S.A. (1996 a 2000)
### Base de Dados: `D-GLOBO.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: Bovespa. Diária bursátil (1.499 pregões). Variáveis: Time, D-GLOBO.
* **Coluna Temporal**: `Time`
* **Variável de Análise Principal**: `D-GLOBO`
* **Parâmetros Sugeridos**: Janela SMA = `21` dias de pregão, Span EWMA = `21` dias de pregão, Período Sazonal $s$ = `21`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `D-GLOBO.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 15: D-GLOBO.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_15 = carregar_serie(
    nome_arquivo="D-GLOBO.csv",
    col_data="Time",
    col_valor="D-GLOBO"
)

print(f"Dataset: D-GLOBO.csv")
print(f"Total de registros válidos: {len(serie_15)}")
print(f"Intervalo temporal: [{serie_15.index[0]}] até [{serie_15.index[-1]}]")
print("Primeiros registros observados:\n", serie_15.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_15,
    janela_sma=21,
    span_ewma=21,
    titulo="Exercício 15 — Suavização Temporal (SMA vs EWMA): Cotações Diárias das Ações da Globo Cabo S.A. (1996 a 2000)",
    ylabel="D-GLOBO",
    unidade="dias de pregão"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_15,
    periodo=21,
    modelo='additive',
    titulo="Exercício 15 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 15):
1. Séries de preços de ações satisfazem a hipótese de passeio aleatório (Random Walk): faz sentido econômico esperar sazonalidade determinística periódica em ativos líquidos?
2. Em termos de estratégias de negociação técnica quantitativa, como o cruzamento do preço de fechamento com a curva EWMA atua como sinalizador de momento (momentum)?
3. Explique a razão pela qual a decomposição clássica concentra a esmagadora maioria da variância da série no componente residual.


---
## Exercício 16: Cotações Diárias das Ações da TAM Linhas Aéreas (1995 a 1999)
### Base de Dados: `D-TAM.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: Bovespa. Diária (1.499 pregões). Variáveis: Time, D-TAM.
* **Coluna Temporal**: `Time`
* **Variável de Análise Principal**: `D-TAM`
* **Parâmetros Sugeridos**: Janela SMA = `30` dias úteis, Span EWMA = `15` dias úteis, Período Sazonal $s$ = `21`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `D-TAM.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 16: D-TAM.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_16 = carregar_serie(
    nome_arquivo="D-TAM.csv",
    col_data="Time",
    col_valor="D-TAM"
)

print(f"Dataset: D-TAM.csv")
print(f"Total de registros válidos: {len(serie_16)}")
print(f"Intervalo temporal: [{serie_16.index[0]}] até [{serie_16.index[-1]}]")
print("Primeiros registros observados:\n", serie_16.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_16,
    janela_sma=30,
    span_ewma=15,
    titulo="Exercício 16 — Suavização Temporal (SMA vs EWMA): Cotações Diárias das Ações da TAM Linhas Aéreas (1995 a 1999)",
    ylabel="D-TAM",
    unidade="dias úteis"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_16,
    periodo=21,
    modelo='additive',
    titulo="Exercício 16 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 16):
1. A distribuição empírica dos resíduos financeiros apresenta homocedasticidade ou agrupamentos de volatilidade (volatility clusters)?
2. Compare a suavização da janela de 30 dias com o span de 15 dias da EWMA durante tendências laterais de preço.
3. Por que modelos da família ARCH/GARCH são mais apropriados para a variância do que a decomposição periódica em séries dessa natureza?


---
## Exercício 17: Série Multiativos Diários da Bovespa: Ibovespa, Petrobras, Banespa e Cemig
### Base de Dados: `day_IBV-PETRO-BANESPA-CEMIG.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: Bovespa (1995 a 1999 — 1.499 pregões). Diária. Variáveis: Time, IBV, PETRO, BANESPA, CEMIG.
* **Coluna Temporal**: `Time`
* **Variável de Análise Principal**: `PETRO`
* **Parâmetros Sugeridos**: Janela SMA = `30` dias úteis, Span EWMA = `14` dias úteis, Período Sazonal $s$ = `21`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `day_IBV-PETRO-BANESPA-CEMIG.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 17: day_IBV-PETRO-BANESPA-CEMIG.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_17 = carregar_serie(
    nome_arquivo="day_IBV-PETRO-BANESPA-CEMIG.csv",
    col_data="Time",
    col_valor="PETRO"
)

print(f"Dataset: day_IBV-PETRO-BANESPA-CEMIG.csv")
print(f"Total de registros válidos: {len(serie_17)}")
print(f"Intervalo temporal: [{serie_17.index[0]}] até [{serie_17.index[-1]}]")
print("Primeiros registros observados:\n", serie_17.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_17,
    janela_sma=30,
    span_ewma=14,
    titulo="Exercício 17 — Suavização Temporal (SMA vs EWMA): Série Multiativos Diários da Bovespa: Ibovespa, Petrobras, Banespa e Cemig",
    ylabel="PETRO",
    unidade="dias úteis"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_17,
    periodo=21,
    modelo='additive',
    titulo="Exercício 17 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 17):
1. Analise o comportamento da tendência de PETRO frente ao choque sistêmico da desvalorização cambial de janeiro de 1999.
2. Altere o argumento para IBV (índice agregado de mercado): a volatilidade residual relativa do índice diversificado é menor que a de uma ação isolada?
3. Qual o papel da EWMA como estimador dinâmico de tendência de curto prazo em sistemas algorítmicos de negociação?


---
## Exercício 18: Índice Ibovespa e Títulos da Dívida Soberana em Frequência Mensal (1994 a 2001)
### Base de Dados: `M-IBV-SP.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: Morettin & Toloi / Banco Central. Mensal (1994 a 2001 — 87 observações). Variáveis: Time, SEP, IBOV, B3, CBOND.
* **Coluna Temporal**: `Time`
* **Variável de Análise Principal**: `IBOV`
* **Parâmetros Sugeridos**: Janela SMA = `12` meses, Span EWMA = `6` meses, Período Sazonal $s$ = `12`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `M-IBV-SP.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 18: M-IBV-SP.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_18 = carregar_serie(
    nome_arquivo="M-IBV-SP.csv",
    col_data="Time",
    col_valor="IBOV"
)

print(f"Dataset: M-IBV-SP.csv")
print(f"Total de registros válidos: {len(serie_18)}")
print(f"Intervalo temporal: [{serie_18.index[0]}] até [{serie_18.index[-1]}]")
print("Primeiros registros observados:\n", serie_18.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_18,
    janela_sma=12,
    span_ewma=6,
    titulo="Exercício 18 — Suavização Temporal (SMA vs EWMA): Índice Ibovespa e Títulos da Dívida Soberana em Frequência Mensal (1994 a 2001)",
    ylabel="IBOV",
    unidade="meses"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_18,
    periodo=12,
    modelo='additive',
    titulo="Exercício 18 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 18):
1. Em frequência mensal agregada, a série do IBOV apresenta indícios de sazonalidade persistente de fim de ano (year-end rally)?
2. Avalie a variável CBOND: em que momentos históricos a tendência dos papéis da dívida sofreu desvalorizações severas?
3. Como a agregação temporal (de diária para mensal) afeta a razão sinal-ruído da série temporal?


---
## Exercício 19: Faturamento Contínuo Horário de Operação de Comércio Eletrônico
### Base de Dados: `Dataset_Ecommerce_Faturamento_Horario.csv`

#### Contexto e Especificações do Processo:
* **Descrição**: Operação de varejo digital (1.001 observações horárias consecutivas). Horária (s=24). Variáveis: data_hora, faturamento_milhares_reais, num_pedidos, ticket_medio_reais, visitas_site.
* **Coluna Temporal**: `data_hora`
* **Variável de Análise Principal**: `faturamento_milhares_reais`
* **Parâmetros Sugeridos**: Janela SMA = `24` horas, Span EWMA = `12` horas, Período Sazonal $s$ = `24`.

#### Roteiro de Tarefas:
1. Carregar a série temporal a partir de `Dataset_Ecommerce_Faturamento_Horario.csv`.
2. Calcular e plotar as Médias Móveis (**SMA** e **EWMA**) com a função `plotar_sma_ewma`.
3. Estimar e plotar a **Decomposição Estrutural** com a função `plotar_decomposicao`.
4. Analisar os gráficos obtidos e responder às questões analíticas propostas.


In [ ]:
# ==============================================================================
# EXERCÍCIO 19: Dataset_Ecommerce_Faturamento_Horario.csv (RESOLUÇÃO)
# ==============================================================================

# 1. Carregamento e sanitização da série temporal
serie_19 = carregar_serie(
    nome_arquivo="Dataset_Ecommerce_Faturamento_Horario.csv",
    col_data="data_hora",
    col_valor="faturamento_milhares_reais"
)

print(f"Dataset: Dataset_Ecommerce_Faturamento_Horario.csv")
print(f"Total de registros válidos: {len(serie_19)}")
print(f"Intervalo temporal: [{serie_19.index[0]}] até [{serie_19.index[-1]}]")
print("Primeiros registros observados:\n", serie_19.head(3))

# 2. Aplicação dos filtros lineares de suavização temporal (SMA e EWMA)
plotar_sma_ewma(
    serie=serie_19,
    janela_sma=24,
    span_ewma=12,
    titulo="Exercício 19 — Suavização Temporal (SMA vs EWMA): Faturamento Contínuo Horário de Operação de Comércio Eletrônico",
    ylabel="faturamento_milhares_reais",
    unidade="horas"
)

# 3. Decomposição canônica da série temporal
plotar_decomposicao(
    serie=serie_19,
    periodo=24,
    modelo='additive',
    titulo="Exercício 19 — Decomposição Estrutural Canônica"
)


#### Questões Analíticas para Avaliação (Exercício 19):
1. Demonstre por que a fixação de k=24 horas para o filtro SMA remove integralmente a variação circadiana e evidencia a tendência de faturamento diário.
2. Analise a componente sazonal horária: quais os intervalos intradiários de pico de faturamento e quais os de vale estrutural (madrugada)?
3. A hipótese de aditividade é satisfatória ou a intensidade das oscilações durante eventos promocionais recomenda uma modelagem multiplicativa?


---
## Síntese Teórica e Diretrizes Estatísticas Finais

Ao concluir o conjunto de exercícios práticos, consolidam-se as seguintes conclusões metodológicas fundamentais:

### 1. Propriedades Comparativas dos Filtros de Suavização:
* **Filtro Retangular (SMA)**: $SMA_k$ confere peso idêntico ($1/k$) a todas as observações no horizonte retroativo. Apresenta propriedade de anulação exata da sazonalidade determinística periódica quando a janela coincide com o ciclo fundamental ($k = s$). Em contrapartida, impõe defasagem de fase proporcional a $(k-1)/2$ diante de tendências aceleradas.
* **Filtro Exponencial (EWMA)**: Atribui pesos $\alpha(1-\alpha)^i$ decrescentes no tempo. Reduz drasticamente a defasagem dinâmica nas vizinhanças de choques e quebras estruturais, constituindo a base dos modelos preditivos adaptativos de Holt-Winters.

### 2. Escolha do Modelo Estrutural de Decomposição:
* **Aditivo**: Recomendado quando a variância sazonal $\text{Var}(S_t)$ é invariante em relação à trajetória da tendência $\mathbb{E}[T_t]$ (homocedasticidade sazonal).
* **Multiplicativo**: Mandatório em séries de crescimento cumulativo onde os desvios cíclicos e o erro percentual escalam proporcionalmente ao nível médio da série.

### 3. Continuidade do Programa de Estudos:
* Diagnóstico de estacionariedade via testes de raiz unitária (**ADF — Augmented Dickey-Fuller** e **KPSS**).
* Identificação da ordem estocástica através das funções de autocorrelação (**ACF**) e autocorrelação parcial (**PACF**).
* Estimação de modelos paramétricos lineares autorregressivos integrados de médias móveis (**ARIMA / SARIMA** e **SARIMAX** com variáveis exógenas).
